In [1]:
# * * * * * * * * * * * * * * * * * * * * * *
# Author      : Robert Meza
# Cohort      : UC Berkeley ML/AI — March 2025
# Description : Capstone Project for CalPERS
# File        : capstone_2_of_2.ipynb
# * * * * * * * * * * * * * * * * * * * * * *

In [3]:
# <- ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! !
# This is part two (capstone_2_of_2.ipynb), 
# the previous steps were completed in 
# part 1 (capstone_1_of_2.ipynb)
# <- ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! !

In [5]:
# start

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import time
import warnings

from scipy import stats
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import tree
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore")

In [8]:
# read the final dataset
df = pd.read_csv("data/final_data.csv")

In [9]:
# sample down for performance
df = df.iloc[:10000]

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   YEAR                       10000 non-null  int64 
 1   AGE                        10000 non-null  int64 
 2   CARRIER                    10000 non-null  object
 3   PREMIUM_MTHLY_AMT          10000 non-null  int64 
 4   DEP_COUNT                  10000 non-null  int64 
 5   HIRE_YEAR                  10000 non-null  int64 
 6   LOV_HLTH_PLAN_TYPE_CD_ASC  10000 non-null  int64 
 7   LOV_HLTH_PLAN_TYPE_CD_EPO  10000 non-null  int64 
 8   LOV_HLTH_PLAN_TYPE_CD_HMO  10000 non-null  int64 
 9   LOV_HLTH_PLAN_TYPE_CD_PPO  10000 non-null  int64 
dtypes: int64(9), object(1)
memory usage: 781.4+ KB


In [12]:
df.head()

,YEAR,AGE,CARRIER,PREMIUM_MTHLY_AMT,DEP_COUNT,HIRE_YEAR,LOV_HLTH_PLAN_TYPE_CD_ASC,LOV_HLTH_PLAN_TYPE_CD_EPO,LOV_HLTH_PLAN_TYPE_CD_HMO,LOV_HLTH_PLAN_TYPE_CD_PPO
0,2021,30,Anthem Blue Cross,2208,4,2018,0,0,0,1
1,2022,46,Blue Shield of California,1958,5,1999,1,0,0,0
2,2021,37,Anthem Blue Cross,1371,2,2011,0,0,0,1
3,2020,29,UnitedHealthcare Services Inc.,1454,1,2015,0,0,1,0
4,2024,40,UnitedHealthcare Services Inc.,2296,3,2018,0,0,1,0


In [17]:
# * * * * * * * * * * * * * * * * * * * * * *
# 5. Classification Modeling
# * * * * * * * * * * * * * * * * * * * * * *

In [19]:
target = "CARRIER"
X = df.drop(columns=[target])

le = LabelEncoder()
y = le.fit_transform(df[target])

In [21]:
# test train split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=6625, stratify=y)

In [23]:
# scale the data after test train split
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

In [25]:
models = {
    "Logistic Regression": LogisticRegression(random_state=6625),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=6625),
    "SVM": SVC()
}

In [27]:
results = []
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    end = time.time()
    
    train_time = end - start
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    
    results.append([name, train_time, train_acc, test_acc])

In [28]:
results = pd.DataFrame(results, columns=["Model", "Train Time", "Train Accuracy", "Test Accuracy"])
results["Train Time"] = results["Train Time"].round(3)
results["Train Accuracy"] = results["Train Accuracy"].round(3)
results["Test Accuracy"] = results["Test Accuracy"].round(3)
print(results)

                 Model  Train Time  Train Accuracy  Test Accuracy
0  Logistic Regression       0.197           0.462          0.462
1                  KNN       0.013           0.934          0.902
2        Decision Tree       0.010           1.000          0.995
3                  SVM       3.454           0.462          0.462


In [29]:
# attempt to improve model performance

In [30]:
# Attempt to tune logistic Regression 

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=6625)
scoring = "f1_weighted"

results1 = []
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(random_state=6625, max_iter=5000))
])

param_lr = {
    "clf__C": [0.001, .01, 0.1, 1, 10, 100, 1000]
}

grid_lr = GridSearchCV(pipe_lr, param_lr, cv=cv, scoring=scoring, n_jobs=-1, refit=True)
grid_lr.fit(X_train, y_train)
y_pred = grid_lr.predict(X_test)

results1.append({
    "Model": "Logistic Regression",
    "Best Params": grid_lr.best_params_,
    "Test Acc": round(accuracy_score(y_test, y_pred), 4)
})

results1


[{'Model': 'Logistic Regression',
  'Best Params': {'clf__C': 100},
  'Test Acc': 0.68}]

In [31]:
# Attempt to tune KNN
results2 = []
param_knn = {
    "clf__n_neighbors": list(range(1, 30, 2)),
    "clf__weights": ["uniform", "distance"],
    "clf__p": [1, 2]
}

pipe_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier(n_jobs=-1))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=6625)

grid_knn = GridSearchCV(
    estimator=pipe_knn,
    param_grid=param_knn,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1,
    refit=True,
    verbose=0
).fit(X_train, y_train)

y_pred = grid_knn.predict(X_test)

results2.append({
    "Model": "KNN",
    "Best Params": grid_knn.best_params_,
    "Test Acc": round(accuracy_score(y_test, y_pred), 4)
})

results2

[{'Model': 'KNN',
  'Best Params': {'clf__n_neighbors': 1,
   'clf__p': 1,
   'clf__weights': 'uniform'},
  'Test Acc': 0.717}]

In [32]:
# Attempt to tune decision tree 
# concern --> the test accuracy was 100%

depths = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30, 40, 50]
results3 = []

for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=6625, class_weight="balanced")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results3.append({
        "Max Depth": d,
        "Train Acc": round(accuracy_score(y_train, model.predict(X_train)), 4),
        "Test Acc": round(accuracy_score(y_test, y_pred), 4),
        "Test F1w": round(f1_score(y_test, y_pred, average="weighted"), 4)
    })

results3 = pd.DataFrame(results3)
results3

,Max Depth,Train Acc,Test Acc,Test F1w
0,1,0.2213,0.2215,0.0825
1,2,0.3041,0.3065,0.2368
2,3,0.3065,0.3110,0.2453
3,4,0.3365,0.3355,0.2786
4,5,0.3205,0.3215,0.3750
5,6,0.3337,0.3345,0.3794
6,7,0.3802,0.3850,0.4668
7,8,0.4168,0.4220,0.5118
8,9,0.5817,0.5855,0.6409
9,10,0.6657,0.6580,0.7354


In [42]:
# Attempt to tubne SVM
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=6625)
scoring = "f1_weighted"

pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(random_state=6625, cache_size=1000)) 
])

param_svm = [
    {"clf__kernel": ["rbf"],
     "clf__C": [0.1, 1, 3, 10, 30],
     "clf__gamma": ["scale", 0.01, 0.1, 1]},
    {"clf__kernel": ["linear"],
     "clf__C": [0.1, 1, 3, 10]}
]

grid_svm = GridSearchCV(
    estimator=pipe_svm,
    param_grid=param_svm,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    refit=True,
    verbose=0
)
grid_svm.fit(X_train, y_train)
y_pred = grid_svm.predict(X_test)

results4 = pd.DataFrame([{
    "Model": "SVM",
    "Best Params": grid_svm.best_params_,
    "Test Acc": round(accuracy_score(y_test, y_pred), 4)
}])

results4

,Model,Best Params,Test Acc
0,SVM,"{'clf__C': 30, 'clf__gamma': 1, 'clf__kernel':...",0.7935


In [123]:
# end